# In vivo temperature measurements analysis across mice

In [1]:
%load_ext autoreload
%autoreload 2

### Imports

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob 

from usnm2p.logger import logger
from usnm2p.constants import *
from usnm2p.thermal_utils import *
from usnm2p.fileops import *

### Inputs

In [23]:
dataroot = '/Users/tlemaire/Documents/data/US_thermal_experiments'

### Load data

In [ ]:

# Compute target ISPTA value corresponding to the reference pressure P_REF and experimental duty cycle DC
DC = 80.
target_ISPPA = pressure_to_intensity(P_REF / PA_TO_MPA) / M2_TO_CM2  # W/cm2
target_ISPTA = np.round(target_ISPPA * DC / 1e2, 2)  # W/cm2

# Load data from CSV files containing temperature elevation values per location at the target ISPTA value
csv_fpaths = glob.glob(
    os.path.join(dataroot, f'max_ΔT_at_ISPTA_{target_ISPTA:.2f}_W_per_cm2_*.csv'))
csv_fnames = [os.path.basename(fpath) for fpath in csv_fpaths]
fpattern = 'max_ΔT_at_ISPTA_(\d+\.\d+)_W_per_cm2_(.*).csv'
ΔT = {}
for fname in csv_fnames:
    mo = re.match(fpattern, fname)
    if mo:
        target_ISPTA_str, experiment_name = mo.groups()
        target_ISPTA_val = float(target_ISPTA_str)
        if np.isclose(target_ISPTA_val, target_ISPTA):
            logger.info(f'loading temperature elevation data from "{fname}"')
            ΔT[experiment_name] = pd.read_csv(os.path.join(dataroot, fname), index_col=0)
ΔT = pd.concat(ΔT, axis=0, names=['experiment'])[REL_TEMP_KEY]
ΔT

### Compute max temperature change at target ISPTA per animal

In [20]:
max_ΔT_per_animal = ΔT.groupby('experiment').max()
max_ΔT_per_animal

experiment
20260910_M01    0.649342
20260911_M02    1.224199
Name: ΔT (°C), dtype: float64

### Compute mean +/- std of max temperature change at target ISPTA across animals

In [22]:
mean_max_ΔT = max_ΔT_per_animal.mean()
std_max_ΔT = max_ΔT_per_animal.std()
logger.info(f'max temperature elevation at {target_ISPTA:.2f} W/cm²: ΔT = {mean_max_ΔT:.2f} ± {std_max_ΔT:.2f} °C (mean ± std across {len(max_ΔT_per_animal)} animals)')

 2026/09/16 16:46:51: max temperature elevation at 15.83 W/cm²: ΔT = 0.94 ± 0.41 °C (mean ± std across 2 animals)
